# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11

In [8]:
%pip install pandas regex razdel pymorphy3

  Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl.metadata (16 kB)
  Using cached razdel-0.5.0-py3-none-any.whl.metadata (10.0 kB)
  Using cached pymorphy3-2.0.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached dawg2_python-0.9.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp313-cp313-win_amd64.whl (10.7 MB)
Using cached razdel-0.5.0-py3-none-any.whl (21 kB)
Using cached pymorphy3-2.0.6-py3-none-any.whl (53 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------ --------------------------- 3.9/12.6 MB 21.6 MB/s eta 0:00:01
   -------------- ------------------------- 4.5/12.6 MB 11.2 MB/s eta 0:00:01
   --------------- ------------------------ 5.0/12.6 MB 7.7 MB/s eta 0:00:01
   ------------------ --------------------- 5.8/12.6 MB 6.7 MB/s eta 0:00:02
   --------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\honor\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [1]:
import os
import regex as re
import pandas as pd

from razdel import tokenize
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

In [3]:
print("раб директория:")
print(os.getcwd())

print("\nФайлы в тек папке:")
print(os.listdir())

раб директория:
C:\Users\honor

Файлы в тек папке:
['.cache', '.ipynb_checkpoints', '.ipython', '.jupyter', '.python_history', '10_vector_sem (1).ipynb', '10_vector_sem.ipynb', '9_text_analysis (1).ipynb', '9_text_analysis.ipynb', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'Desktop', 'Documents', 'Downloads', 'erdman.txt', 'Favorites', 'hw3.ipynb', 'hw4.ipynb', 'Links', 'Local Settings', 'Music', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TM.blf', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'Pictures', 'PrintHood', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'vam_paintings.csv', 'Videos', 'главное меню', 'Мои документы', 'Шаблоны']


In [5]:
with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

print("Колво символов:", len(text))
print("\nНачало текста:")
print(text[:1000])

Колво символов: 88123

Начало текста:
[Воспоминания А. И. Степановой]
Шел 1928 год. Я была замужем за Николаем Михайловичем Горчаковым, педагогом и режиссером Художественного театра. К этому времени я прошла трудный и счастливый путь артистки знаменитого театра; репетировала и играла со Станиславским, Книппер-Чеховой, Качаловым, Леонидовым, Москвиным, Тархановым, Кореневой, Халютиной. На моем счету уже значилось шесть больших ролей. Мы с мужем жили в Кривоарбатском переулке, дом наш — одну большую комнату — друзья любили за тепло и гостеприимство. К нам часто заходили коллеги-мхатовцы, писатели, художники. Всегда находилось, чем угостить их или просто накормить обедом, ужином. Частыми гостями были в доме Марков, Бабель, неразлучные тогда Олеша и Катаев, художники Дмитриев и Вильямс. Заходил и замечательный актер Театра сатиры, обаятельный Владимир Яковлевич Хенкин, остроумный, неутомимый рассказчик. Несколько раз мы принимали у себя Всеволода Эмильевича Мейерхольда с Зинаидой Райх и об

In [6]:
text_clean = re.sub(r"\[\d+\]", "", text)
text_clean = text_clean.replace("\r\n", "\n").replace("\r", "\n")

print("Исходный текст:", len(text), "символов")
print("После очистки:", len(text_clean), "символов")

Исходный текст: 88123 символов
После очистки: 88076 символов


In [7]:
blocks = re.split(
    r"(?m)^\s*\*\s*\*\s*\*\s*$",
    text_clean
)

blocks = [block.strip() for block in blocks if block.strip()]

print("Колво блоков:", len(blocks))

for i, block in enumerate(blocks[:5]):
    print(f"\n--- Блок {i} ---")
    print(block[:500])

Колво блоков: 75

--- Блок 0 ---
[Воспоминания А. И. Степановой]
Шел 1928 год. Я была замужем за Николаем Михайловичем Горчаковым, педагогом и режиссером Художественного театра. К этому времени я прошла трудный и счастливый путь артистки знаменитого театра; репетировала и играла со Станиславским, Книппер-Чеховой, Качаловым, Леонидовым, Москвиным, Тархановым, Кореневой, Халютиной. На моем счету уже значилось шесть больших ролей. Мы с мужем жили в Кривоарбатском переулке, дом наш — одну большую комнату — друзья любили за тепло и 

--- Блок 1 ---
Баку,
Оперный театр. Гастроли М.Х.Т.,
А. О. Степановой.
Через несколько дней после вашего отъезда я заперся в своей комнате, снял пиджак и ботинки и с головой нырнул в пьесу. Только вчера я из нее вынырнул. За все это время я ни разу не вышел из дому. В отвратительной бороде я сидел за столом, поклявшись всем дорогим на свете, а значит, и вами, Лина, не вставать из-за него, пока не поставлю последней точки. Вот почему я так поздно получил ваши пи

In [8]:
letter_blocks = blocks[1:]

print("Колво блоков после исключения:",
      len(letter_blocks))

Колво блоков после исключения: 74


In [9]:
letter_blocks[-1] = re.sub(
    r"Они встретились в квартире Бориса Робертовича в 1957 году.*$",
    "",
    letter_blocks[-1],
    flags=re.DOTALL
).strip()

letters = []

for block in letter_blocks:
    first_lines = "\n".join(block.splitlines()[:5]).strip()

    if re.match(r"(?i)^телеграмма\b", first_lines):
        continue

    letters.append(block)

print("Количество писем:", len(letters))

Количество писем: 73


In [10]:
for i, letter in enumerate(letters):
    print(f"\n{'=' * 60}")
    print(f"письмо {i}")
    print(letter[:700])


письмо 0
Баку,
Оперный театр. Гастроли М.Х.Т.,
А. О. Степановой.
Через несколько дней после вашего отъезда я заперся в своей комнате, снял пиджак и ботинки и с головой нырнул в пьесу. Только вчера я из нее вынырнул. За все это время я ни разу не вышел из дому. В отвратительной бороде я сидел за столом, поклявшись всем дорогим на свете, а значит, и вами, Лина, не вставать из-за него, пока не поставлю последней точки. Вот почему я так поздно получил ваши письма и вот почему я так долго не отвечал. Много раз я отрывался от пьесы и принимался за письмо к вам, но, не дописавши его, принимался за пьесу. Кончалось это тем, что я в совершенном отчаянье рвал и то, и другое. Простите меня, милая, я очень винов

письмо 1
Батум,
Чаква, Цихис-Дзири,
Ольге Алексеевне Ростовцевой — для А. О. Степановой.
Мне было очень хорошо с Вами. Я никогда не забуду Ваших утренних появлений со стаканом чая в руках и Ваших таинственных исчезновений ночью, о которых я узнавал только на другой день.
Я никогда не заб

In [11]:
months = {
    "январ": 1,
    "феврал": 2,
    "март": 3,
    "марта": 3,
    "апрел": 4,
    "мая": 5,
    "май": 5,
    "июн": 6,
    "июл": 7,
    "август": 8,
    "сентябр": 9,
    "октябр": 10,
    "ноябр": 11,
    "декабр": 12
}

roman_months = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
    "VII": 7,
    "VIII": 8,
    "IX": 9,
    "X": 10,
    "XI": 11,
    "XII": 12
}


def normalize_year(year):
    if year is None:
        return None

    year = int(year)

    if year < 100:
        return 1900 + year

    return year


def parse_date(letter):
    

    header = "\n".join(letter.splitlines()[:15])

    
    match = re.search(
        r"\b(\d{1,2})[.](\d{1,2})[.](\d{2,4})\b",
        header
    )

    if match:
        day, month, year = match.groups()

        return {
            "day": int(day),
            "month": int(month),
            "year": normalize_year(year),
            "date_raw": match.group(0),
            "date_type": "explicit"
        }

  
    match = re.search(
        r"\b(\d{1,2})\s*/\s*([IVX]+|\d{1,2})\s*[-–]\s*(\d{2,4})\b",
        header
    )

    if match:
        day, month_raw, year = match.groups()

        if month_raw.isdigit():
            month = int(month_raw)
        else:
            month = roman_months.get(month_raw.upper())

        if month is not None and 1 <= month <= 12:
            return {
                "day": int(day),
                "month": month,
                "year": normalize_year(year),
                "date_raw": match.group(0),
                "date_type": "explicit"
            }

    
    month_pattern = (
        r"(январ[ьяе]|феврал[ьяе]|март[а]?|апрел[ьяе]|мая|май|"
        r"июн[ьяе]|июл[ьяе]|август[а]?|сентябр[ьяе]|"
        r"октябр[ьяе]|ноябр[ьяе]|декабр[ьяе])"
    )

    match = re.search(
        r"\b(\d{1,2})\s+" + month_pattern +
        r"(?:\s+(\d{2,4})\s*(?:г\.?|года)?)?",
        header,
        flags=re.IGNORECASE
    )

    if match:
        day = int(match.group(1))
        month_raw = match.group(2).lower()
        year = match.group(3)

        month = None

        for key, value in months.items():
            if month_raw.startswith(key):
                month = value
                break

        if month is not None:
            return {
                "day": day,
                "month": month,
                "year": normalize_year(year) if year else None,
                "date_raw": match.group(0),
                "date_type": "explicit" if year else "day_month"
            }


    for key, value in months.items():
        match = re.search(
            r"(?im)^\s*(" + key + r"[а-я]*)\.?\s*$",
            header
        )

        if match:
            return {
                "day": None,
                "month": value,
                "year": None,
                "date_raw": match.group(0).strip(),
                "date_type": "month_only"
            }

    return {
        "day": None,
        "month": None,
        "year": None,
        "date_raw": None,
        "date_type": "missing"
    }

In [12]:
date_data = []

for i, letter in enumerate(letters):
    result = parse_date(letter)
    result["letter_id"] = i
    date_data.append(result)

dates_df = pd.DataFrame(date_data)

dates_df.head(15)

,day,month,year,date_raw,date_type,letter_id
0,NaN,NaN,NaN,NaN,missing,0
1,NaN,NaN,NaN,NaN,missing,1
2,3.0,11.0,1933.0,3 ноября 1933 г.,explicit,2
3,4.0,11.0,1933.0,4 ноября 1933 г,explicit,3
4,5.0,11.0,NaN,5 ноября,day_month,4
5,6.0,11.0,1933.0,6 ноября 1933 г,explicit,5
6,6.0,11.0,NaN,6 ноября,day_month,6
7,16.0,11.0,NaN,16 ноября,day_month,7
8,20.0,11.0,1933.0,20 ноября 1933 г.,explicit,8
9,22.0,11.0,1933.0,22 ноября 1933 г.,explicit,9


In [13]:
dates_df["year_final"] = dates_df["year"]
dates_df["year_source"] = dates_df["year"].apply(
    lambda x: "explicit" if pd.notna(x) else "not_specified"
)

explicit_year_indices = dates_df[
    dates_df["year"].notna()
].index.tolist()


for i in dates_df.index:

   
    if pd.notna(dates_df.loc[i, "year"]):
        continue

    
    if pd.isna(dates_df.loc[i, "month"]):
        continue

    
    previous = [
        j for j in explicit_year_indices if j < i
    ]

    following = [
        j for j in explicit_year_indices if j > i
    ]

    previous_index = previous[-1] if previous else None
    following_index = following[0] if following else None

    if previous_index is not None and following_index is not None:

        distance_previous = i - previous_index
        distance_following = following_index - i

        if distance_previous <= distance_following:
            inferred_year = int(dates_df.loc[previous_index, "year"])
        else:
            inferred_year = int(dates_df.loc[following_index, "year"])

    elif previous_index is not None:
        inferred_year = int(dates_df.loc[previous_index, "year"])

    elif following_index is not None:
        inferred_year = int(dates_df.loc[following_index, "year"])

    else:
        continue

    dates_df.loc[i, "year_final"] = inferred_year
    dates_df.loc[i, "year_source"] = "inferred"


print("Способ опред года:")
print(dates_df["year_source"].value_counts())

Способ опред года:
year_source
inferred         31
explicit         24
not_specified    18
Name: count, dtype: int64


In [14]:
from datetime import date


def make_iso_date(row):

    day = row["day"]
    month = row["month"]
    year = row["year_final"]

    if pd.isna(day) or pd.isna(month) or pd.isna(year):
        return None

    try:
        return date(
            int(year),
            int(month),
            int(day)
        ).isoformat()

    except ValueError:
        return None


dates_df["date"] = dates_df.apply(make_iso_date, axis=1)

display(
    dates_df[
        ["letter_id", "date_raw", "date", "date_type", "year_source"]
    ].head(20)
)

,letter_id,date_raw,date,date_type,year_source
0,0,NaN,NaN,missing,not_specified
1,1,NaN,NaN,missing,not_specified
2,2,3 ноября 1933 г.,1933-11-03,explicit,explicit
3,3,4 ноября 1933 г,1933-11-04,explicit,explicit
4,4,5 ноября,1933-11-05,day_month,inferred
5,5,6 ноября 1933 г,1933-11-06,explicit,explicit
6,6,6 ноября,1933-11-06,day_month,inferred
7,7,16 ноября,1933-11-16,day_month,inferred
8,8,20 ноября 1933 г.,1933-11-20,explicit,explicit
9,9,22 ноября 1933 г.,1933-11-22,explicit,explicit


In [15]:
print("без полной даты:")
display(
    dates_df[dates_df["date"].isna()][
        ["letter_id", "date_raw", "date_type", "year_source"]
    ]
)

без полной даты:


,letter_id,date_raw,date_type,year_source
0,0,NaN,missing,not_specified
1,1,NaN,missing,not_specified
18,18,NaN,missing,not_specified
19,19,NaN,missing,not_specified
20,20,NaN,missing,not_specified
22,22,NaN,missing,not_specified
24,24,NaN,missing,not_specified
26,26,NaN,missing,not_specified
27,27,NaN,missing,not_specified
28,28,NaN,missing,not_specified


In [16]:
def detect_author(letter):

    lines = [
        line.strip()
        for line in letter.splitlines()
        if line.strip()
    ]

    header = "\n".join(lines[:15])
    ending = "\n".join(lines[-8:])

   
    to_stepanova = re.search(
        r"(Ангелин[еы]\s+Осиповн[еы]\s+Степанов[аоы]?|"
        r"А\.\s*О\.\s*Степанов[аоы]?)",
        header,
        flags=re.IGNORECASE
    )

   
    to_erdman = re.search(
        r"(Николаю\s+Робертовичу\s+Эрдману|"
        r"Николаю\s+Эрдману)",
        header,
        flags=re.IGNORECASE
    )

    
    signature_erdman = re.search(
        r"(?im)^\s*(Николай|Н\.?\s*Р\.?\s*Эрдман)\.?\s*$",
        ending
    )

    signature_stepanova = re.search(
        r"(?im)^\s*(Лина|Линушка|Ангелина)\.?\s*$",
        ending
    )

    if to_stepanova and not to_erdman:
        return "Николай Эрдман", "recipient"

    if to_erdman and not to_stepanova:
        return "Ангелина Степанова", "recipient"

    if signature_erdman and not signature_stepanova:
        return "Николай Эрдман", "signature"

    if signature_stepanova and not signature_erdman:
        return "Ангелина Степанова", "signature"

    
    return None, "manual_check"

In [17]:
author_data = []

for i, letter in enumerate(letters):

    author, method = detect_author(letter)

    author_data.append({
        "letter_id": i,
        "author": author,
        "author_method": method
    })


authors_df = pd.DataFrame(author_data)

display(authors_df.head(20))

,letter_id,author,author_method
0,0,Николай Эрдман,recipient
1,1,Николай Эрдман,recipient
2,2,Николай Эрдман,recipient
3,3,Ангелина Степанова,signature
4,4,Ангелина Степанова,signature
5,5,Николай Эрдман,signature
6,6,Ангелина Степанова,signature
7,7,Ангелина Степанова,signature
8,8,Николай Эрдман,recipient
9,9,NaN,manual_check


In [18]:
unknown_authors = authors_df[
    authors_df["author"].isna()
]

print("Коливо писем с неопр автором:",
      len(unknown_authors))

for i in unknown_authors["letter_id"]:
    print(f"\n{'=' * 50}")
    print(f"ПИСЬМО {i}")
    print(letters[i][:1200])

Коливо писем с неопр автором: 11

ПИСЬМО 9
22 ноября 1933 г.
Вот это жизнь — сюрприз за сюрпризом. Сегодня решил зайти к своим картавцам и отнести им чего-нибудь сладенького. К моим «конфектам» они уже привыкли, поэтому я вооружился ножом и стал открывать деревянную коробку, которая была в Твоей посылке. Остальное Тебе понятно. Скажу только, что папирос у меня почти не осталось. Несмотря на мои сомнения, дети жевали «конфекты» с прежним энтузиазмом и были на седьмом небе, а я курил «Герцеговину» и был ничуть не ниже их. Сегодня был в Райпрофсовете и в Райкоме партии, предлагал им себя. Вероятно, буду работать в клубе. В клубе изредка играет единственная в городе труппа любителей. Передай мхатовским актрисам, если им нечего играть — пусть приезжают, я поставлю «Марию Стюарт» или «Орлеанскую деву».
Начал писать. В связи с пьесой у меня к Тебе просьба. Видишь, как часто стал я Тебе надоедать. Узнай, пожалуйста, у Мамошина или у кого-нибудь еще, как пишутся заявления о приеме в партию. Пиш

In [19]:
author_corrections = {
    9: "Николай Эрдман",
    21: "Николай Эрдман",
    25: "Николай Эрдман",
    26: "Николай Эрдман",
    32: "Николай Эрдман",
    33: "Ангелина Степанова",
    34: "Николай Эрдман",
    35: "Ангелина Степанова",
    38: "Николай Эрдман",
    62: "Николай Эрдман",
    71: "Николай Эрдман"
}

for letter_id, author in author_corrections.items():

    authors_df.loc[
        authors_df["letter_id"] == letter_id,
        "author"
    ] = author

    authors_df.loc[
        authors_df["letter_id"] == letter_id,
        "author_method"
    ] = "manual"

In [20]:
def extract_body(letter):

    lines = letter.splitlines()


    for i, line in enumerate(lines):

        if not line.strip():
            body = "\n".join(lines[i + 1:]).strip()

            if body:
                return body

 
    return letter.strip()


letter_texts = [
    extract_body(letter)
    for letter in letters
]

In [21]:
clean_texts = []

for letter in letter_texts:


    letter = re.sub(r"\s+", " ", letter)

    
    letter = letter.strip()

    clean_texts.append(letter)


print(clean_texts[0][:1000])

Баку, Оперный театр. Гастроли М.Х.Т., А. О. Степановой. Через несколько дней после вашего отъезда я заперся в своей комнате, снял пиджак и ботинки и с головой нырнул в пьесу. Только вчера я из нее вынырнул. За все это время я ни разу не вышел из дому. В отвратительной бороде я сидел за столом, поклявшись всем дорогим на свете, а значит, и вами, Лина, не вставать из-за него, пока не поставлю последней точки. Вот почему я так поздно получил ваши письма и вот почему я так долго не отвечал. Много раз я отрывался от пьесы и принимался за письмо к вам, но, не дописавши его, принимался за пьесу. Кончалось это тем, что я в совершенном отчаянье рвал и то, и другое. Простите меня, милая, я очень виноват перед вами. Бог знает, сколько грустного вы передумали, не получая моих писем. Может быть, вы даже подумали, что я позабыл вас. Если вы так подумали, то вы тоже виноваты передо мной. Сегодня, по дороге к Массу, я проходил по Кривоарбатскому переулку и смотрел в ваши окна. До чего же мне было тоск

In [22]:
def lemmatize(text):

    
    tokens = [
        token.text.lower()
        for token in tokenize(text)
    ]

    lemmas = []

    for token in tokens:

        if token.isalnum():

            lemma = morph.parse(token)[0].normal_form

            lemmas.append(lemma)

    return " ".join(lemmas)

In [23]:
lemmatized_texts = [
    lemmatize(letter)
    for letter in clean_texts
]

print("Нач текст:")
print(clean_texts[0][:500])

print("\nЛемматизированный:")
print(lemmatized_texts[0][:500])

Нач текст:
Баку, Оперный театр. Гастроли М.Х.Т., А. О. Степановой. Через несколько дней после вашего отъезда я заперся в своей комнате, снял пиджак и ботинки и с головой нырнул в пьесу. Только вчера я из нее вынырнул. За все это время я ни разу не вышел из дому. В отвратительной бороде я сидел за столом, поклявшись всем дорогим на свете, а значит, и вами, Лина, не вставать из-за него, пока не поставлю последней точки. Вот почему я так поздно получил ваши письма и вот почему я так долго не отвечал. Много ра

Лемматизированный:
баку оперный театр гастроль м х так а о степанов через несколько день после ваш отъезд я запереться в свой комната снять пиджак и ботинок и с голова нырнуть в пьеса только вчера я из она вынырнуть за всё это время я ни раз не выйти из дом в отвратительный борода я сидеть за стол поклясться весь дорогой на свет а значит и вы лина не вставать он пока не поставить последний точка вот почему я так поздно получить ваш письмо и вот почему я так долго не отвечать много р

In [24]:
def count_words(text):

    tokens = [
        token.text
        for token in tokenize(text)
    ]

    
    words = [
        token for token in tokens
        if token.isalnum()
    ]

    return len(words)

In [25]:
word_counts = [
    count_words(letter)
    for letter in clean_texts
]

print("Колво слов в первом письме:", word_counts[0])

Колво слов в первом письме: 216


In [26]:
df = pd.DataFrame({
    "letter_id": range(len(letters)),
    "text": clean_texts,
    "lemmatized_text": lemmatized_texts,
    "word_count": word_counts
})


df = df.merge(
    dates_df[["letter_id", "date"]],
    on="letter_id",
    how="left"
)


df = df.merge(
    authors_df[["letter_id", "author"]],
    on="letter_id",
    how="left"
)


df = df[
    ["date", "author", "text", "lemmatized_text", "word_count"]
]

df.head()

,date,author,text,lemmatized_text,word_count
0,NaN,Николай Эрдман,"Баку, Оперный театр. Гастроли М.Х.Т., А. О. Ст...",баку оперный театр гастроль м х так а о степан...,216
1,NaN,Николай Эрдман,"Батум, Чаква, Цихис-Дзири, Ольге Алексеевне Ро...",батум чаква ольга алексеевич ростовцев для а о...,505
2,1933-11-03,Николай Эрдман,"Москва, проезд Художественного театра, Художес...",москва проезд художественный театр художествен...,133
3,1933-11-04,Ангелина Степанова,"4 ноября 1933 года Дорогой, прекрасный, любимы...",4 ноябрь 1933 год дорога прекрасный любимый я ...,126
4,1933-11-05,Ангелина Степанова,"5 ноября Дорогой мой, посылка халтурно составл...",5 ноябрь дорога мой посылка халтурно составить...,30


In [27]:
print("Размер:")
print(df.shape)

print("\nDataFrame:")
df.info()

print("\nПервые строки:")
display(df.head())

print("\nКолво пропусков:")
print(df.isna().sum())

Размер:
(73, 5)

DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   date             54 non-null     str  
 1   author           73 non-null     str  
 2   text             73 non-null     str  
 3   lemmatized_text  73 non-null     str  
 4   word_count       73 non-null     int64
dtypes: int64(1), str(4)
memory usage: 3.0 KB

Первые строки:


,date,author,text,lemmatized_text,word_count
0,NaN,Николай Эрдман,"Баку, Оперный театр. Гастроли М.Х.Т., А. О. Ст...",баку оперный театр гастроль м х так а о степан...,216
1,NaN,Николай Эрдман,"Батум, Чаква, Цихис-Дзири, Ольге Алексеевне Ро...",батум чаква ольга алексеевич ростовцев для а о...,505
2,1933-11-03,Николай Эрдман,"Москва, проезд Художественного театра, Художес...",москва проезд художественный театр художествен...,133
3,1933-11-04,Ангелина Степанова,"4 ноября 1933 года Дорогой, прекрасный, любимы...",4 ноябрь 1933 год дорога прекрасный любимый я ...,126
4,1933-11-05,Ангелина Степанова,"5 ноября Дорогой мой, посылка халтурно составл...",5 ноябрь дорога мой посылка халтурно составить...,30



Колво пропусков:
date               19
author              0
text                0
lemmatized_text     0
word_count          0
dtype: int64


In [28]:
print("Колво писем каждого автора:")
print(df["author"].value_counts(dropna=False))

Колво писем каждого автора:
author
Николай Эрдман        42
Ангелина Степанова    31
Name: count, dtype: int64


In [29]:
print("Колво с датой:")
print(df["date"].notna().sum())

print("Колво без полной даты:")
print(df["date"].isna().sum())

Колво с датой:
54
Колво без полной даты:
19


In [30]:
print("Пустых текст:", (df["text"].str.strip() == "").sum())
print(
    "Пустых леммат текстов:",
    (df["lemmatized_text"].str.strip() == "").sum()
)

print("Самое короткое:", df["word_count"].min())
print("Самое длинное:", df["word_count"].max())

print("Повтов:", df["text"].duplicated().sum())

Пустых текст: 0
Пустых леммат текстов: 0
Самое короткое: 9
Самое длинное: 648
Повтов: 0


In [31]:
df.to_csv(
    "erdman_letters.csv",
    index=False,
    encoding="utf-8-sig"
)

In [32]:
print(os.listdir())

['.cache', '.ipynb_checkpoints', '.ipython', '.jupyter', '.python_history', '10_vector_sem (1).ipynb', '10_vector_sem.ipynb', '9_text_analysis (1).ipynb', '9_text_analysis.ipynb', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'Desktop', 'Documents', 'Downloads', 'erdman.txt', 'erdman_letters.csv', 'Favorites', 'hw3.ipynb', 'hw4.ipynb', 'Links', 'Local Settings', 'Music', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TM.blf', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{f74e6495-9fe4-11f1-b524-eae4baa9ed4e}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'Pictures', 'PrintHood', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'vam_paintings.csv', 'Videos', 'главное меню', 'Мои документы', 'Шаблоны']
